# Fused Leaky ReLU forward and gradient

Leaky ReLU keeps a small slope $\alpha$ for negative inputs so their gradient never vanishes completely:

$$y = \begin{cases} x & x > 0 \\ \alpha x & x \le 0 \end{cases} \qquad \frac{dy}{dx} = \begin{cases} 1 & x > 0 \\ \alpha & x \le 0 \end{cases}$$

Training needs both, and reading `x` once to produce both is cheaper than two separate kernels. Write `leaky_relu(x, alpha)` that launches **one** kernel writing two outputs, and returns the tuple `(y, dydx)`.

## Inputs and outputs
- `x`: 1-D `float32` array of length `n` (possibly 0). `alpha`: Python float (may be 0 or negative; just apply the formula).
- `y`, `dydx`: `float32` arrays of shape `(n,)`.
- At exactly `x == 0` use the second branch: `y = 0`, `dydx = alpha`.

## Launch
128 threads per block, $\lceil n/128 \rceil$ blocks. For `n == 0` return two empty `float32` arrays without launching. `alpha` is passed to the kernel as an ordinary scalar argument.

## Runtime
GPU in Colab; checked here on Numba's CUDA simulator.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
!pip -q install "numba-cuda[cu12]"

In [ ]:
import numpy as np
from numba import cuda

TPB = 128

def leaky_relu(x, alpha):
    """x: float32 (n,), alpha: float. Returns (y, dydx), both float32 (n,), from one kernel launch."""
    n = x.shape[0]
    y = np.zeros(n, dtype=np.float32)
    dydx = np.zeros(n, dtype=np.float32)
    # TODO: allocate both outputs on the device and launch leaky_kernel once
    return y, dydx

@cuda.jit
def leaky_kernel(x, alpha, y, dydx):
    # TODO
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [-2.0, 0.0, 3.0], "dtype": "float32"}, 0.1], "e": {"__tup__": [{"__nd__": [-0.20000000298023224, 0.0, 3.0], "dtype": "float32"}, {"__nd__": [0.10000000149011612, 0.10000000149011612, 1.0], "dtype": "float32"}]}}, {"a": [{"__nd__": [1.0, -1.0, -0.5, 0.25], "dtype": "float32"}, 0.01], "e": {"__tup__": [{"__nd__": [1.0, -0.009999999776482582, -0.004999999888241291, 0.25], "dtype": "float32"}, {"__nd__": [1.0, 0.009999999776482582, 0.009999999776482582, 1.0], "dtype": "float32"}]}}, {"a": [{"__nd__": [], "dtype": "float32"}, 0.2], "e": {"__tup__": [{"__nd__": [], "dtype": "float32"}, {"__nd__": [], "dtype": "float32"}]}}, {"a": [{"__nd__": [0.0, 0.0, 0.0, 0.0], "dtype": "float32"}, 0.3], "e": {"__tup__": [{"__nd__": [0.0, 0.0, 0.0, 0.0], "dtype": "float32"}, {"__nd__": [0.30000001192092896, 0.30000001192092896, 0.30000001192092896, 0.30000001192092896], "dtype": "float32"}]}}, {"a": [{"__nd__": [-5.0, 5.0], "dtype": "float32"}, 0.0], "e": {"__tup__": [{"__nd__": [-0.0, 5.0], "dtype": "float32"}, {"__nd__": [0.0, 1.0], "dtype": "float32"}]}}, {"a": [{"__nd__": [1.053115725517273, 1.7764912843704224, -2.5532917976379395, -0.13796506822109222, 1.0137194395065308, 1.3521418571472168, 0.6537883877754211, 1.4971178770065308, 0.28995758295059204, 0.551267147064209, 0.17873768508434296, -1.0738587379455566, -0.8466289639472961, 0.37958425283432007, -0.5801951885223389, 1.2715513706207275, 1.2923866510391235, 1.7987862825393677, -0.026073837652802467, 1.3837097883224487, -0.9058431386947632, -0.8163147568702698, 0.08130305260419846, 0.28143084049224854, -1.5989882946014404, -1.7307242155075073, 0.35507839918136597, -0.8621881008148193, 1.2083570957183838, 0.3945462107658386, 0.3071846067905426, 0.20844262838363647, 0.8954750895500183, -0.1530461460351944, 1.085631012916565, -0.5259143114089966, -0.22499771416187286, -0.6805052161216736, 0.5976618528366089, 0.03588424623012543, -0.8622670769691467, 2.001879930496216, -0.3637140393257141, -0.11629977077245712, 0.3941647708415985, 1.7476508617401123, -0.14838363230228424, 0.7031832337379456, 0.0987953394651413, -1.4637936353683472, 0.7919772863388062, 1.0206393003463745, -0.23998627066612244, -0.01543633732944727, -1.5013161897659302, -1.5229519605636597, -0.3915199339389801, 0.6259333491325378, -0.46957460045814514, 0.2920228838920593, 0.9677673578262329, 0.09576239436864853, 0.5744683742523193, 1.4478734731674194, -1.2460280656814575, 0.7440750598907471, -1.8846845626831055, 0.4316425919532776, 0.20299464464187622, -0.5433347821235657, -1.0773130655288696, -0.06183105707168579, -0.410238653421402, 0.09762159734964371, -0.9078017473220825, 0.5455273389816284, -1.0358502864837646, -0.3042352795600891, -0.391988068819046, 1.2678874731063843, -1.4956938028335571, -0.41402381658554077, 1.1824606657028198, -0.6583412289619446, -1.2394710779190063, 0.2828414738178253, -0.29003041982650757, 0.8347549438476562, -0.9203901290893555, 0.19091922044754028, 0.6661273241043091, 1.0357218980789185, -1.1061452627182007, 1.4237887859344482, -0.9006966352462769, -1.5125505924224854, -0.9545692205429077, 0.15484929084777832, -0.321784108877182, 0.4404556155204773, 0.4956032633781433, 0.3181445002555847, -0.25452789664268494, 1.8140779733657837, -2.216033458709717, -1.734349012374878, -0.5068859457969666, -1.2552173137664795, -2.5559253692626953, -0.7950685024261475, -0.7492610812187195, -2.448549270629883, 0.8586929440498352, 1.29911208152771, 1.2343010902404785, -0.5083407163619995, 0.14761216938495636, -1.1208102703094482, 0.20643730461597443, -0.6741753816604614, 0.4098832309246063, -0.3429722785949707, 1.0337574481964111, 0.8266077637672424, 1.23894464969635, -1.6052957773208618, -1.8196072578430176, 0.4047423303127289, -0.5771601796150208, 0.3978289067745209, 0.29888084530830383, 0.18021494150161743, 0.9192535281181335, 0.556616485118866, 0.4724460244178772, -0.8938198685646057, 1.7246652841567993, 0.12661360204219818, 1.1070493459701538, 0.32706713676452637, 1.2693654298782349, 0.15353532135486603, 0.2176106721162796, 0.13781337440013885, -0.7086793184280396, -0.5720164775848389, -0.14694882929325104, -0.11698143184185028, -0.9826943278312683, 2.356566905975342, -0.9813703894615173, -0.30599862337112427, 2.4452872276306152, -0.6596879363059998, 0.7659382820129395, 0.6608838438987732, -0.2080206423997879, -1.4289289712905884, -1.553024411201477, -0.00791721697896719, 0.4208160936832428, -2.1561195850372314, 0.15284812450408936, -0.654123067855835, 0.8952039480209351, 0.17636170983314514, -0.8014308214187622, -0.42353519797325134, 0.048485130071640015, 1.3498164415359497, 3.0509796142578125, 0.36641550064086914, 0.1300293207168579, 0.7127132415771484, 0.045923635363578796, 0.7104279398918152, 0.4690491259098053, 0.08745628595352173, -0.7064696550369263, 0.24971434473991394, 0.994346022605896, 0.16790692508220673, 1.608073353767395, 1.0373547077178955, -0.03474288433790207, 1.8357981443405151, -1.0261919498443604, 1.2709004878997803, 1.9574341773986816, -0.15091410279273987, 0.0018051289953291416, 1.0439820289611816, 1.5392496585845947, 0.5528356432914734, 0.3550683259963989, 1.033794641494751, 0.3781958520412445, -1.0895569324493408, -0.0690082460641861, 0.1042112335562706, -1.3806716203689575, 0.518618643283844, -0.6232043504714966, 0.861382782459259, -1.3710020780563354, -0.7937610149383545, -1.3396885395050049, 1.4022269248962402, 1.3917750120162964, 1.5862153768539429, 1.5932339429855347, 1.0219627618789673, 1.0119965076446533, -1.5809804201126099, 0.4466247856616974, -0.3821673095226288, -0.20327728986740112, -1.1104215383529663, 1.431710958480835, 0.27714720368385315, -1.3678317070007324, 0.6259223818778992, 0.03098929673433304, -0.9454757571220398, -0.21463562548160553, -0.700574517250061, -2.2325522899627686, -1.4735822677612305, 0.5461509823799133, 0.3729606568813324, 0.2895790934562683, -1.166759967803955, -0.5945008993148804, 0.06568827480077744, 0.6906121373176575, -0.8430119752883911, -1.059950590133667, 1.1398041248321533, 0.06655634194612503, -0.3809283673763275, -0.7401725649833679, -0.6576257348060608, 1.5846054553985596, 0.9693785905838013, -0.4177705943584442, -1.2538760900497437, 1.6734874248504639, 1.5702478885650635, 1.2041453123092651, 0.8415461182594299, -1.887092113494873, -1.2911195755004883, 0.5649392604827881, -0.28491517901420593, 0.2554653584957123, -0.5254474878311157, -0.6570103168487549, 0.5533406734466553, -0.4006863832473755, -0.002750141080468893, 0.25537389516830444, 0.2316483110189438, 1.158631682395935, 1.36827552318573, -1.3684428930282593, -0.2694779336452484, -0.6488304138183594, 0.7532873153686523, -1.0702744722366333, 0.7732221484184265, 0.6983717679977417, 0.5756646990776062, -0.9760543704032898, -0.21768690645694733, -1.0833427906036377, 0.39390647411346436, 0.9520752429962158, -1.5239372253417969, -0.8299877047538757, -0.16650493443012238, -0.5342420935630798, 1.4900349378585815, -0.14836764335632324, 0.2496156245470047, -1.1146836280822754, 1.5401816368103027, 0.16285106539726257, 0.7711102366447449, 0.08412633091211319, -2.1234958171844482, -1.600638747215271, -1.2906688451766968, 0.4687395989894867, -0.5121772885322571, 0.42633765935897827, 0.284776896238327, 0.2474629282951355, 0.9171476364135742, 0.18053820729255676, -1.6436583995819092], "dtype": "float32"}, 0.2], "e": {"__tup__": [{"__nd__": [1.053115725517273, 1.7764912843704224, -0.5106583833694458, -0.027593014761805534, 1.0137194395065308, 1.3521418571472168, 0.6537883877754211, 1.4971178770065308, 0.28995758295059204, 0.551267147064209, 0.17873768508434296, -0.21477174758911133, -0.1693257987499237, 0.37958425283432007, -0.11603903770446777, 1.2715513706207275, 1.2923866510391235, 1.7987862825393677, -0.0052147675305604935, 1.3837097883224487, -0.18116863071918488, -0.16326294839382172, 0.08130305260419846, 0.28143084049224854, -0.31979766488075256, -0.3461448550224304, 0.35507839918136597, -0.1724376231431961, 1.2083570957183838, 0.3945462107658386, 0.3071846067905426, 0.20844262838363647, 0.8954750895500183, -0.03060922957956791, 1.085631012916565, -0.10518286377191544, -0.04499954357743263, -0.13610105216503143, 0.5976618528366089, 0.03588424623012543, -0.17245341837406158, 2.001879930496216, -0.07274281233549118, -0.023259954527020454, 0.3941647708415985, 1.7476508617401123, -0.02967672608792782, 0.7031832337379456, 0.0987953394651413, -0.2927587330341339, 0.7919772863388062, 1.0206393003463745, -0.04799725487828255, -0.0030872675124555826, -0.30026325583457947, -0.3045904040336609, -0.0783039852976799, 0.6259333491325378, -0.09391491860151291, 0.2920228838920593, 0.9677673578262329, 0.09576239436864853, 0.5744683742523193, 1.4478734731674194, -0.24920561909675598, 0.7440750598907471, -0.3769369125366211, 0.4316425919532776, 0.20299464464187622, -0.10866695642471313, -0.2154626101255417, -0.012366211973130703, -0.0820477306842804, 0.09762159734964371, -0.18156035244464874, 0.5455273389816284, -0.2071700543165207, -0.06084705516695976, -0.07839761674404144, 1.2678874731063843, -0.29913875460624695, -0.08280476182699203, 1.1824606657028198, -0.13166825473308563, -0.24789421260356903, 0.2828414738178253, -0.05800608545541763, 0.8347549438476562, -0.18407802283763885, 0.19091922044754028, 0.6661273241043091, 1.0357218980789185, -0.22122906148433685, 1.4237887859344482, -0.18013933300971985, -0.3025101125240326, -0.1909138411283493, 0.15484929084777832, -0.06435682624578476, 0.4404556155204773, 0.4956032633781433, 0.3181445002555847, -0.050905581563711166, 1.8140779733657837, -0.44320669770240784, -0.3468697965145111, -0.10137718915939331, -0.2510434687137604, -0.5111851096153259, -0.15901370346546173, -0.1498522162437439, -0.48970985412597656, 0.8586929440498352, 1.29911208152771, 1.2343010902404785, -0.10166814178228378, 0.14761216938495636, -0.2241620570421219, 0.20643730461597443, -0.13483507931232452, 0.4098832309246063, -0.06859445571899414, 1.0337574481964111, 0.8266077637672424, 1.23894464969635, -0.3210591673851013, -0.36392146348953247, 0.4047423303127289, -0.11543203890323639, 0.3978289067745209, 0.29888084530830383, 0.18021494150161743, 0.9192535281181335, 0.556616485118866, 0.4724460244178772, -0.1787639707326889, 1.7246652841567993, 0.12661360204219818, 1.1070493459701538, 0.32706713676452637, 1.2693654298782349, 0.15353532135486603, 0.2176106721162796, 0.13781337440013885, -0.14173586666584015, -0.11440329998731613, -0.029389766976237297, -0.023396287113428116, -0.19653886556625366, 2.356566905975342, -0.19627408683300018, -0.061199724674224854, 2.4452872276306152, -0.13193759322166443, 0.7659382820129395, 0.6608838438987732, -0.04160412773489952, -0.2857857942581177, -0.31060490012168884, -0.0015834433725103736, 0.4208160936832428, -0.43122392892837524, 0.15284812450408936, -0.13082461059093475, 0.8952039480209351, 0.17636170983314514, -0.16028617322444916, -0.08470704406499863, 0.048485130071640015, 1.3498164415359497, 3.0509796142578125, 0.36641550064086914, 0.1300293207168579, 0.7127132415771484, 0.045923635363578796, 0.7104279398918152, 0.4690491259098053, 0.08745628595352173, -0.14129392802715302, 0.24971434473991394, 0.994346022605896, 0.16790692508220673, 1.608073353767395, 1.0373547077178955, -0.006948576774448156, 1.8357981443405151, -0.20523838698863983, 1.2709004878997803, 1.9574341773986816, -0.030182821676135063, 0.0018051289953291416, 1.0439820289611816, 1.5392496585845947, 0.5528356432914734, 0.3550683259963989, 1.033794641494751, 0.3781958520412445, -0.21791139245033264, -0.01380164921283722, 0.1042112335562706, -0.27613434195518494, 0.518618643283844, -0.12464087456464767, 0.861382782459259, -0.2742004096508026, -0.1587522029876709, -0.26793771982192993, 1.4022269248962402, 1.3917750120162964, 1.5862153768539429, 1.5932339429855347, 1.0219627618789673, 1.0119965076446533, -0.316196084022522, 0.4466247856616974, -0.076433464884758, -0.040655460208654404, -0.22208431363105774, 1.431710958480835, 0.27714720368385315, -0.273566335439682, 0.6259223818778992, 0.03098929673433304, -0.1890951544046402, -0.042927127331495285, -0.1401149034500122, -0.4465104639530182, -0.2947164475917816, 0.5461509823799133, 0.3729606568813324, 0.2895790934562683, -0.23335199058055878, -0.11890017986297607, 0.06568827480077744, 0.6906121373176575, -0.16860239207744598, -0.2119901180267334, 1.1398041248321533, 0.06655634194612503, -0.0761856734752655, -0.14803451299667358, -0.13152514398097992, 1.5846054553985596, 0.9693785905838013, -0.08355411887168884, -0.25077521800994873, 1.6734874248504639, 1.5702478885650635, 1.2041453123092651, 0.8415461182594299, -0.3774184286594391, -0.25822392106056213, 0.5649392604827881, -0.05698303505778313, 0.2554653584957123, -0.10508950054645538, -0.13140206038951874, 0.5533406734466553, -0.08013727515935898, -0.0005500282277353108, 0.25537389516830444, 0.2316483110189438, 1.158631682395935, 1.36827552318573, -0.27368858456611633, -0.05389558896422386, -0.1297660917043686, 0.7532873153686523, -0.2140548974275589, 0.7732221484184265, 0.6983717679977417, 0.5756646990776062, -0.19521087408065796, -0.043537382036447525, -0.21666856110095978, 0.39390647411346436, 0.9520752429962158, -0.30478745698928833, -0.16599754989147186, -0.033300988376140594, -0.10684841871261597, 1.4900349378585815, -0.029673529788851738, 0.2496156245470047, -0.2229367345571518, 1.5401816368103027, 0.16285106539726257, 0.7711102366447449, 0.08412633091211319, -0.42469915747642517, -0.3201277554035187, -0.25813376903533936, 0.4687395989894867, -0.10243546217679977, 0.42633765935897827, 0.284776896238327, 0.2474629282951355, 0.9171476364135742, 0.18053820729255676, -0.3287316858768463], "dtype": "float32"}, {"__nd__": [1.0, 1.0, 0.20000000298023224, 0.20000000298023224, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.20000000298023224, 0.20000000298023224, 1.0, 0.20000000298023224, 1.0, 1.0, 1.0, 0.20000000298023224, 1.0, 0.20000000298023224, 0.20000000298023224, 1.0, 1.0, 0.20000000298023224, 0.20000000298023224, 1.0, 0.20000000298023224, 1.0, 1.0, 1.0, 1.0, 1.0, 0.20000000298023224, 1.0, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 1.0, 1.0, 0.20000000298023224, 1.0, 0.20000000298023224, 0.20000000298023224, 1.0, 1.0, 0.20000000298023224, 1.0, 1.0, 0.20000000298023224, 1.0, 1.0, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 1.0, 0.20000000298023224, 1.0, 1.0, 1.0, 1.0, 1.0, 0.20000000298023224, 1.0, 0.20000000298023224, 1.0, 1.0, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 1.0, 0.20000000298023224, 1.0, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 1.0, 0.20000000298023224, 0.20000000298023224, 1.0, 0.20000000298023224, 0.20000000298023224, 1.0, 0.20000000298023224, 1.0, 0.20000000298023224, 1.0, 1.0, 1.0, 0.20000000298023224, 1.0, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 1.0, 0.20000000298023224, 1.0, 1.0, 1.0, 0.20000000298023224, 1.0, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 1.0, 1.0, 1.0, 0.20000000298023224, 1.0, 0.20000000298023224, 1.0, 0.20000000298023224, 1.0, 0.20000000298023224, 1.0, 1.0, 1.0, 0.20000000298023224, 0.20000000298023224, 1.0, 0.20000000298023224, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.20000000298023224, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 1.0, 0.20000000298023224, 0.20000000298023224, 1.0, 0.20000000298023224, 1.0, 1.0, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 1.0, 0.20000000298023224, 1.0, 0.20000000298023224, 1.0, 1.0, 0.20000000298023224, 0.20000000298023224, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.20000000298023224, 1.0, 1.0, 1.0, 1.0, 1.0, 0.20000000298023224, 1.0, 0.20000000298023224, 1.0, 1.0, 0.20000000298023224, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.20000000298023224, 0.20000000298023224, 1.0, 0.20000000298023224, 1.0, 0.20000000298023224, 1.0, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.20000000298023224, 1.0, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 1.0, 1.0, 0.20000000298023224, 1.0, 1.0, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 1.0, 1.0, 1.0, 0.20000000298023224, 0.20000000298023224, 1.0, 1.0, 0.20000000298023224, 0.20000000298023224, 1.0, 1.0, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 1.0, 1.0, 0.20000000298023224, 0.20000000298023224, 1.0, 1.0, 1.0, 1.0, 0.20000000298023224, 0.20000000298023224, 1.0, 0.20000000298023224, 1.0, 0.20000000298023224, 0.20000000298023224, 1.0, 0.20000000298023224, 0.20000000298023224, 1.0, 1.0, 1.0, 1.0, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 1.0, 0.20000000298023224, 1.0, 1.0, 1.0, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 1.0, 1.0, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 1.0, 0.20000000298023224, 1.0, 0.20000000298023224, 1.0, 1.0, 1.0, 1.0, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 1.0, 0.20000000298023224, 1.0, 1.0, 1.0, 1.0, 1.0, 0.20000000298023224], "dtype": "float32"}]}}, {"a": [{"__nd__": [-0.6808697581291199, 1.6592670679092407, -1.0736650228500366, 1.3619978427886963, 1.652419090270996, -0.595105767250061, 0.34424883127212524, -1.4023863077163696, 0.7722803354263306, -0.7778273224830627, -1.3104488849639893, -0.39303863048553467, -0.4573878347873688, -0.5853225588798523, 0.19626294076442719, -0.16898725926876068, -0.6634804010391235, 0.9487982392311096, 2.421513795852661, 0.06504317373037338, -0.704403281211853, -0.992775022983551, 0.5899581909179688, -0.25655415654182434, 0.015056869015097618, -1.9947081804275513, -1.1320641040802002, -1.162248134613037, 0.943668007850647, -0.2703447639942169, -2.4855313301086426, 0.4818969666957855, 1.1324982643127441, 1.3041704893112183, -2.8018319606781006, 0.15273961424827576, 0.8111956715583801, 0.6056417226791382, -0.23600471019744873, -1.8013386726379395, -0.4336090087890625, -0.32346421480178833, 0.5738714337348938, 0.6397247910499573, -0.24265602231025696, 0.8471004366874695, -1.05661940574646, -0.6826376914978027, -0.031049536541104317, 0.573591411113739, -0.32215219736099243, 1.2306468486785889, 0.41090700030326843, -0.9491457939147949, 0.08558788895606995, -0.8875037431716919, -0.9625638723373413, -1.42739737033844, 1.397571086883545, 0.9094139933586121, -1.600577473640442, 1.1129155158996582, 0.9609569907188416, -0.3633536696434021, 0.5071889162063599, 2.244964361190796, 0.2862309515476227, 0.4218981862068176, 1.1223537921905518, -1.0346606969833374, -1.2659227848052979, -1.4094098806381226, -0.07717163860797882, 0.8727633953094482, -0.8587529063224792, -1.3691868782043457, 0.21724383533000946, -0.34941476583480835, 0.6297116875648499, -0.09816327691078186, -0.1489013135433197, -0.0463363453745842, 0.07550323754549026, -0.9222617149353027, -1.0816967487335205, 1.631859540939331, 3.1339569091796875, 1.1654694080352783, -1.5833669900894165, 0.07792814075946808, 1.2387341260910034, -1.5705281496047974, -0.43609005212783813, -0.35702264308929443, -0.8635562658309937, -0.7844062447547913, 1.0680334568023682, -0.39739489555358887, -0.17256353795528412, 1.7736471891403198, -0.7939521074295044, -1.8595759868621826, 0.7050238251686096, 2.070965528488159, 0.2678645849227905, 0.36346182227134705, -1.2320520877838135, -0.8820333480834961, 1.5223171710968018, 1.316070795059204, 0.6436273455619812, -0.5896125435829163, -1.1411457061767578, -0.9983368515968323, -1.128322720527649, 0.655798614025116, -0.7499189972877502, -0.4616090953350067, 0.7248115539550781, 0.7008728981018066, 0.07221673429012299, 0.4980469346046448, 0.020505748689174652, -1.9255032539367676, -0.06928952038288116, -0.4177575409412384, -0.351639986038208, 0.245271697640419, -0.5644857287406921], "dtype": "float32"}, -0.5], "e": {"__tup__": [{"__nd__": [0.34043487906455994, 1.6592670679092407, 0.5368325114250183, 1.3619978427886963, 1.652419090270996, 0.2975528836250305, 0.34424883127212524, 0.7011931538581848, 0.7722803354263306, 0.38891366124153137, 0.6552244424819946, 0.19651931524276733, 0.2286939173936844, 0.29266127943992615, 0.19626294076442719, 0.08449362963438034, 0.33174020051956177, 0.9487982392311096, 2.421513795852661, 0.06504317373037338, 0.3522016406059265, 0.4963875114917755, 0.5899581909179688, 0.12827707827091217, 0.015056869015097618, 0.9973540902137756, 0.5660320520401001, 0.5811240673065186, 0.943668007850647, 0.13517238199710846, 1.2427656650543213, 0.4818969666957855, 1.1324982643127441, 1.3041704893112183, 1.4009159803390503, 0.15273961424827576, 0.8111956715583801, 0.6056417226791382, 0.11800235509872437, 0.9006693363189697, 0.21680450439453125, 0.16173210740089417, 0.5738714337348938, 0.6397247910499573, 0.12132801115512848, 0.8471004366874695, 0.52830970287323, 0.34131884574890137, 0.015524768270552158, 0.573591411113739, 0.16107609868049622, 1.2306468486785889, 0.41090700030326843, 0.47457289695739746, 0.08558788895606995, 0.44375187158584595, 0.48128193616867065, 0.71369868516922, 1.397571086883545, 0.9094139933586121, 0.800288736820221, 1.1129155158996582, 0.9609569907188416, 0.18167683482170105, 0.5071889162063599, 2.244964361190796, 0.2862309515476227, 0.4218981862068176, 1.1223537921905518, 0.5173303484916687, 0.6329613924026489, 0.7047049403190613, 0.03858581930398941, 0.8727633953094482, 0.4293764531612396, 0.6845934391021729, 0.21724383533000946, 0.17470738291740417, 0.6297116875648499, 0.04908163845539093, 0.07445065677165985, 0.0231681726872921, 0.07550323754549026, 0.46113085746765137, 0.5408483743667603, 1.631859540939331, 3.1339569091796875, 1.1654694080352783, 0.7916834950447083, 0.07792814075946808, 1.2387341260910034, 0.7852640748023987, 0.21804502606391907, 0.17851132154464722, 0.4317781329154968, 0.39220312237739563, 1.0680334568023682, 0.19869744777679443, 0.08628176897764206, 1.7736471891403198, 0.3969760537147522, 0.9297879934310913, 0.7050238251686096, 2.070965528488159, 0.2678645849227905, 0.36346182227134705, 0.6160260438919067, 0.44101667404174805, 1.5223171710968018, 1.316070795059204, 0.6436273455619812, 0.29480627179145813, 0.5705728530883789, 0.49916842579841614, 0.5641613602638245, 0.655798614025116, 0.3749594986438751, 0.23080454766750336, 0.7248115539550781, 0.7008728981018066, 0.07221673429012299, 0.4980469346046448, 0.020505748689174652, 0.9627516269683838, 0.03464476019144058, 0.2088787704706192, 0.175819993019104, 0.245271697640419, 0.28224286437034607], "dtype": "float32"}, {"__nd__": [-0.5, 1.0, -0.5, 1.0, 1.0, -0.5, 1.0, -0.5, 1.0, -0.5, -0.5, -0.5, -0.5, -0.5, 1.0, -0.5, -0.5, 1.0, 1.0, 1.0, -0.5, -0.5, 1.0, -0.5, 1.0, -0.5, -0.5, -0.5, 1.0, -0.5, -0.5, 1.0, 1.0, 1.0, -0.5, 1.0, 1.0, 1.0, -0.5, -0.5, -0.5, -0.5, 1.0, 1.0, -0.5, 1.0, -0.5, -0.5, -0.5, 1.0, -0.5, 1.0, 1.0, -0.5, 1.0, -0.5, -0.5, -0.5, 1.0, 1.0, -0.5, 1.0, 1.0, -0.5, 1.0, 1.0, 1.0, 1.0, 1.0, -0.5, -0.5, -0.5, -0.5, 1.0, -0.5, -0.5, 1.0, -0.5, 1.0, -0.5, -0.5, -0.5, 1.0, -0.5, -0.5, 1.0, 1.0, 1.0, -0.5, 1.0, 1.0, -0.5, -0.5, -0.5, -0.5, -0.5, 1.0, -0.5, -0.5, 1.0, -0.5, -0.5, 1.0, 1.0, 1.0, 1.0, -0.5, -0.5, 1.0, 1.0, 1.0, -0.5, -0.5, -0.5, -0.5, 1.0, -0.5, -0.5, 1.0, 1.0, 1.0, 1.0, 1.0, -0.5, -0.5, -0.5, -0.5, 1.0, -0.5], "dtype": "float32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(leaky_relu, json.loads(_TESTS), 'cuda', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")